In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import torch
import nibabel as nib
import os
import tensorflow as tf
from tensorflow.keras.utils import to_categorical # type: ignore # type: ignore
from skimage.transform import resize


In [2]:
print(pd.__version__)
print(np.__version__)
print(tf.__version__)
print(torch.__version__)
print("GPU is", "available" if torch.cuda.is_available() else "NOT AVAILABLE")

2.2.3
2.0.2
2.18.0
2.6.0+cpu
GPU is NOT AVAILABLE


In [3]:
import os
import numpy as np
import nibabel as nib
from skimage.transform import resize
from tensorflow.keras.utils import to_categorical

VOLUME_DIR = "train_folder"
SEGMENTATION_DIR = os.path.join("train_folder", "Segmentation")
IMG_SIZE = (128, 128)

def load_nifti(file_path):
    """Load a NIfTI file and return the NumPy array"""
    print(f"Loading file: {file_path}") 
    nifti_img = nib.load(file_path) 
    return nifti_img.get_fdata()

def preprocess_data(volume_path, segmentation_path):
    """Load and preprocess volume and segmentation data"""
    volume_data = load_nifti(volume_path)
    segmentation_data = load_nifti(segmentation_path).astype(int)

    volume_data = (volume_data - np.min(volume_data)) / (np.max(volume_data) - np.min(volume_data))

    volume_resized = np.array([
        resize(slice, IMG_SIZE, mode='constant', preserve_range=True) 
        for slice in volume_data.transpose(2, 0, 1)
    ])
    segmentation_resized = np.array([
        resize(slice, IMG_SIZE, mode='constant', preserve_range=True, order=0) 
        for slice in segmentation_data.transpose(2, 0, 1)
    ])

    segmentation_onehot = to_categorical(segmentation_resized, num_classes=3)

    return volume_resized, segmentation_onehot

X_train, Y_train = [], []

image_files = sorted([f for f in os.listdir(VOLUME_DIR) if f.endswith(".nii")])
mask_files = sorted([f for f in os.listdir(SEGMENTATION_DIR) if f.endswith(".nii")])

for img_file, mask_file in zip(image_files, mask_files):
    volume_path = os.path.join(VOLUME_DIR, img_file)
    segmentation_path = os.path.join(SEGMENTATION_DIR, mask_file)
    vol, seg = preprocess_data(volume_path, segmentation_path)
    X_train.append(vol)
    Y_train.append(seg)

X_train = np.concatenate(X_train, axis=0)
Y_train = np.concatenate(Y_train, axis=0)

print("Training data shape:", X_train.shape)
print("Segmentation mask shape:", Y_train.shape)

Loading file: train_folder\volume-0.nii
Loading file: train_folder\Segmentation\segmentation-0.nii
Loading file: train_folder\volume-1.nii
Loading file: train_folder\Segmentation\segmentation-1.nii
Loading file: train_folder\volume-2.nii
Loading file: train_folder\Segmentation\segmentation-2.nii
Loading file: train_folder\volume-3.nii
Loading file: train_folder\Segmentation\segmentation-3.nii
Loading file: train_folder\volume-4.nii
Loading file: train_folder\Segmentation\segmentation-4.nii
Training data shape: (2090, 128, 128)
Segmentation mask shape: (2090, 128, 128, 3)


In [4]:
import os
import numpy as np
import nibabel as nib
from tensorflow.keras.utils import to_categorical # type: ignore
# Function to load NIfTI files
def load_nifti(file_path):
    """Load a NIfTI file and return the NumPy array"""
    print(f"Loading: {file_path}")  # Debugging: Print file being loaded
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    
    nifti_img = nib.load(file_path)
    return nifti_img.get_fdata()

# Function to preprocess test data
def preprocess_data(test_volume_path, test_segmentation_path):
    """Load and preprocess a single test volume and its corresponding segmentation mask"""
    volume_data = load_nifti(test_volume_path)
    segmentation_data = load_nifti(test_segmentation_path).astype(int)

    # Normalize CT images (0-1)
    volume_data = (volume_data - np.min(volume_data)) / (np.max(volume_data) - np.min(volume_data))

    # Resize to (128, 128)
    from skimage.transform import resize
    IMG_SIZE = (128, 128)
    
    volume_resized = np.array([
        resize(slice, IMG_SIZE, mode='constant', preserve_range=True)
        for slice in volume_data.transpose(2, 0, 1)  # Rearrange to (slices, H, W)
    ])
    
    segmentation_resized = np.array([
        resize(slice, IMG_SIZE, mode='constant', preserve_range=True, order=0) 
        for slice in segmentation_data.transpose(2, 0, 1)
    ])

    # One-hot encode segmentation masks (background=0, liver=1, tumor=2)
    
    segmentation_onehot = to_categorical(segmentation_resized, num_classes=3)

    return volume_resized, segmentation_onehot

# ---- Load Test Data ----
test_image_path = os.path.join("test_folder") 
test_mask_path = os.path.join("test_folder", "segmentation") 

# Ensure directories exist
if not os.path.exists(test_image_path) or not os.path.exists(test_mask_path):
    raise FileNotFoundError("Test data folder or segmentation folder not found!")

# Get sorted filenames
image_files = sorted([f for f in os.listdir(test_image_path) if f.endswith(".nii")])
mask_files = sorted([f for f in os.listdir(test_mask_path) if f.endswith(".nii")])

# Check if the number of images and masks match
if len(image_files) != len(mask_files):
    raise ValueError("Mismatch between the number of test images and segmentation masks!")

X_test, Y_test = [], []

# Load and preprocess each test image and mask
for img_file, mask_file in zip(image_files, mask_files):
    volume_path = os.path.join(test_image_path, img_file)  # Correct path joining
    segmentation_path = os.path.join(test_mask_path, mask_file)  # Correct path joining

    vol, seg = preprocess_data(volume_path, segmentation_path)
    X_test.append(vol)
    Y_test.append(seg)

# Convert lists to NumPy arrays
X_test = np.array(X_test)  # Shape: (num_volumes, num_slices, 128, 128)
Y_test = np.array(Y_test)  # Shape: (num_volumes, num_slices, 128, 128, 3)

# Reshape to flatten across all slices
X_test = X_test.reshape(-1, 128, 128, 1)  # Add channel dimension
Y_test = Y_test.reshape(-1, 128, 128, 3)  # Keep segmentation masks in one-hot format

# Print shapes
print("Final Test Data Shape:", X_test.shape)  # (total_slices, 128, 128, 1)
print("Final Test Mask Shape:", Y_test.shape)  # (total_slices, 128, 128, 3)


Loading: test_folder\volume-10.nii
Loading: test_folder\segmentation\segmentation-10.nii
Final Test Data Shape: (501, 128, 128, 1)
Final Test Mask Shape: (501, 128, 128, 3)


In [5]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

# 3. Define the UNet++ model (same as in your original message)
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class UNetPlusPlus(nn.Module):
    def __init__(self, in_channels=1, out_channels=3, filters=[32, 64, 128, 256, 512]):
        super().__init__()
        self.conv0_0 = ConvBlock(in_channels, filters[0])
        self.pool0 = nn.MaxPool2d(2)
        self.conv1_0 = ConvBlock(filters[0], filters[1])
        self.pool1 = nn.MaxPool2d(2)
        self.conv2_0 = ConvBlock(filters[1], filters[2])
        self.pool2 = nn.MaxPool2d(2)
        self.conv3_0 = ConvBlock(filters[2], filters[3])
        self.pool3 = nn.MaxPool2d(2)
        self.conv4_0 = ConvBlock(filters[3], filters[4])

        self.up1_0 = nn.ConvTranspose2d(filters[1], filters[0], kernel_size=2, stride=2)
        self.conv0_1 = ConvBlock(filters[0]*2, filters[0])
        self.up2_0 = nn.ConvTranspose2d(filters[2], filters[1], kernel_size=2, stride=2)
        self.conv1_1 = ConvBlock(filters[1]*2, filters[1])
        self.up3_0 = nn.ConvTranspose2d(filters[3], filters[2], kernel_size=2, stride=2)
        self.conv2_1 = ConvBlock(filters[2]*2, filters[2])
        self.up4_0 = nn.ConvTranspose2d(filters[4], filters[3], kernel_size=2, stride=2)
        self.conv3_1 = ConvBlock(filters[3]*2, filters[3])

        self.up1_1 = nn.ConvTranspose2d(filters[1], filters[0], kernel_size=2, stride=2)
        self.conv0_2 = ConvBlock(filters[0]*3, filters[0])
        self.up2_1 = nn.ConvTranspose2d(filters[2], filters[1], kernel_size=2, stride=2)
        self.conv1_2 = ConvBlock(filters[1]*3, filters[1])
        self.up3_1 = nn.ConvTranspose2d(filters[3], filters[2], kernel_size=2, stride=2)
        self.conv2_2 = ConvBlock(filters[2]*3, filters[2])

        self.final_conv = nn.Conv2d(filters[0], out_channels, kernel_size=1)

    def forward(self, x):
        x0_0 = self.conv0_0(x)
        x1_0 = self.conv1_0(self.pool0(x0_0))
        x2_0 = self.conv2_0(self.pool1(x1_0))
        x3_0 = self.conv3_0(self.pool2(x2_0))
        x4_0 = self.conv4_0(self.pool3(x3_0))

        x3_1 = self.conv3_1(torch.cat([self.up4_0(x4_0), x3_0], 1))
        x2_1 = self.conv2_1(torch.cat([self.up3_0(x3_0), x2_0], 1))
        x1_1 = self.conv1_1(torch.cat([self.up2_0(x2_0), x1_0], 1))
        x0_1 = self.conv0_1(torch.cat([self.up1_0(x1_0), x0_0], 1))

        x2_2 = self.conv2_2(torch.cat([self.up3_1(x3_1), x2_0, x2_1], 1))
        x1_2 = self.conv1_2(torch.cat([self.up2_1(x2_1), x1_0, x1_1], 1))
        x0_2 = self.conv0_2(torch.cat([self.up1_1(x1_1), x0_0, x0_1], 1))

        return self.final_conv(x0_2)



In [6]:
import torch
from torch.utils.data import DataLoader, Dataset

# Assuming x_train is (2090, 128, 128) and y_train is (2090, 128, 128, 3)
class SliceDataset(Dataset):
    def __init__(self, x, y):
        super().__init__()
        # Convert to float32 and torch tensors
        self.x = torch.tensor(x, dtype=torch.float32).unsqueeze(1)  # (N, 1, 128, 128)
        self.y = torch.tensor(y, dtype=torch.float32).permute(0, 3, 1, 2)  # (N, 3, 128, 128)

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

# Create dataset and loader
batch_size = 16
dataset = SliceDataset(X_train, Y_train)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [7]:
# Fetch one batch from the DataLoader and print details
for images, masks in train_loader:
    print("✅ Images shape:", images.shape)  # Should be (16, 1, 128, 128)
    print("✅ Masks shape:", masks.shape)    # Should be (16, 3, 128, 128)
    
    # Check value ranges
    print("Image min/max:", images.min().item(), images.max().item())
    print("Mask unique values (flattened):", torch.unique(masks))

    # Optional: check one-hot encoding validity
    is_one_hot = torch.all((masks.sum(dim=1) == 1) | (masks.sum(dim=1) == 0))
    print("Is one-hot encoded?", is_one_hot.item())

    break  # Just check the first batch

# Test DataLoader
for batch_idx, (images, masks) in enumerate(train_loader):
    print(f"Batch {batch_idx + 1}")
    print(f"Image batch shape: {images.shape}")  # Should be (16, 1, 128, 128)
    print(f"Mask batch shape: {masks.shape}")    # Should be (16, 3, 128, 128)
    
    # Optionally test forward pass through your model
    model = UNetPlusPlus(in_channels=1, out_channels=3)  # Ensure model is defined
    outputs = model(images)  # Forward pass
    print(f"Model output shape: {outputs.shape}")  # Should be (16, 3, 128, 128)
    
    break  # Just check one batch

✅ Images shape: torch.Size([16, 1, 128, 128])
✅ Masks shape: torch.Size([16, 3, 128, 128])
Image min/max: 0.0 0.8935009241104126
Mask unique values (flattened): tensor([0., 1.])
Is one-hot encoded? True
Batch 1
Image batch shape: torch.Size([16, 1, 128, 128])
Mask batch shape: torch.Size([16, 3, 128, 128])
Model output shape: torch.Size([16, 3, 128, 128])


In [8]:
# Print shapes
X_test = np.squeeze(X_test, axis=-1)  # Now shape: (501, 128, 128)
print("Final Test Data Shape:", X_test.shape)  # (total_slices, 128, 128, 1)
print("Final Test Mask Shape:", Y_test.shape)  # (total_slices, 128, 128, 3)
print("Training data shape:", X_train.shape)
print("Segmentation mask shape:", Y_train.shape)

Final Test Data Shape: (501, 128, 128)
Final Test Mask Shape: (501, 128, 128, 3)
Training data shape: (2090, 128, 128)
Segmentation mask shape: (2090, 128, 128, 3)


In [18]:
class TestSliceDataset(Dataset):
    def __init__(self, x, y):
        super().__init__()
        # Remove the last dimension if it's singleton (1)
        if x.shape[-1] == 1:
            x = np.squeeze(x, axis=-1)  # Now (N, 128, 128)

        self.x = torch.tensor(x, dtype=torch.float32).unsqueeze(1)  # (N, 1, 128, 128)
        self.y = torch.tensor(y, dtype=torch.float32).permute(0, 3, 1, 2)  # (N, 3, 128, 128)

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

# Use it
test_dataset = TestSliceDataset(X_test, Y_test)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
for images, masks in test_loader:
    print("Fixed input shape:", images.shape)  # ✅ Should be (16, 1, 128, 128)
    print("Fixed mask shape:", masks.shape)    # ✅ Should be (16, 3, 128, 128)
    
    # Forward pass
    outputs = model(images)
    print("Model output shape:", outputs.shape)
    break


Fixed input shape: torch.Size([16, 1, 128, 128])
Fixed mask shape: torch.Size([16, 3, 128, 128])
Model output shape: torch.Size([16, 3, 128, 128])


In [ ]:
# # 4. Training Setup
# device = "cuda" if torch.cuda.is_available() else "cpu"
# model = UNetPlusPlus().to(device)
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=0.01)

# # 5. Training Loop
# num_epochs = 15
# for epoch in range(num_epochs):
#     model.train()
#     epoch_loss = 0.0
#     for inputs, targets in train_loader:
#         inputs = inputs.to(device)
#         targets = targets.to(device)
#         targets = torch.argmax(targets, dim=1)  # Convert one-hot to class indices

#         outputs = model(inputs)
#         loss = criterion(outputs, targets)

#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()
#         epoch_loss += loss.item()

#     print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {epoch_loss / len(train_loader):.4f}")

In [9]:
# Recreate the model architecture first
model = UNetPlusPlus(in_channels=1, out_channels=3)

# Load the weights
model.load_state_dict(torch.load('Unet_Plus/unetplusplus_epoch_15.pth'))

# Put it in eval mode if you're using it for inference
model.eval()

UNetPlusPlus(
  (conv0_0): ConvBlock(
    (double_conv): Sequential(
      (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (pool0): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv1_0): ConvBlock(
    (double_conv): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inp

In [11]:
def dice_score(preds, targets, epsilon=1e-6):
    # Assumes preds are one-hot or softmax outputs (batch, C, H, W)
    preds = torch.argmax(preds, dim=1)  # (batch, H, W)
    targets = torch.argmax(targets, dim=1)  # (batch, H, W)

    dice_total = 0
    for cls in range(3):  # For each class
        pred_cls = (preds == cls).float()
        target_cls = (targets == cls).float()
        
        intersection = (pred_cls * target_cls).sum()
        union = pred_cls.sum() + target_cls.sum()
        
        dice = (2. * intersection + epsilon) / (union + epsilon)
        dice_total += dice

    return dice_total / 3  # Average over 3 classes

total_dice = 0
num_batches = 0

with torch.no_grad():
    for images, masks in test_loader:
        outputs = model(images)  # Forward pass
        batch_dice = dice_score(outputs, masks)
        total_dice += batch_dice.item()
        num_batches += 1

avg_dice = total_dice / num_batches
print(f"Average Dice Score on Test Set: {avg_dice:.4f}")

Average Dice Score on Test Set: 0.7693


Dice Score Interpretation
Dice Score	Quality	Notes
0.90 – 1.0	Excellent	Near-perfect segmentation
0.80 – 0.90	Good	Acceptable for many tasks
0.70 – 0.80	Moderate	Needs tuning, but decent start
< 0.70	Low	Likely underfitting or dataset/model issue

In [12]:
def dice_score_per_class(preds, targets, epsilon=1e-6):
    # Convert from softmax/one-hot to label maps
    preds = torch.argmax(preds, dim=1)    # (batch, H, W)
    targets = torch.argmax(targets, dim=1)  # (batch, H, W)

    class_dice_scores = []

    for cls in range(3):  # 3 classes
        pred_cls = (preds == cls).float()
        target_cls = (targets == cls).float()

        intersection = (pred_cls * target_cls).sum()
        union = pred_cls.sum() + target_cls.sum()

        dice = (2. * intersection + epsilon) / (union + epsilon)
        class_dice_scores.append(dice.item())

    return class_dice_scores  # List: [dice_class_0, dice_class_1, dice_class_2]

In [13]:
total_dice = [0.0, 0.0, 0.0]
num_batches = 0

with torch.no_grad():
    for images, masks in test_loader:
        outputs = model(images)
        dice_scores = dice_score_per_class(outputs, masks)
        for i in range(3):
            total_dice[i] += dice_scores[i]
        num_batches += 1

avg_dice_per_class = [d / num_batches for d in total_dice]
for i, score in enumerate(avg_dice_per_class):
    print(f"Average Dice Score for Class {i}: {score:.4f}")

Average Dice Score for Class 0: 0.9919
Average Dice Score for Class 1: 0.6284
Average Dice Score for Class 2: 0.6875


#### Class	Dice Score	Comment
0 (Background or dominant class)	0.9919  The model is almost perfect at segmenting this class. This could be because it's the most represented or easiest to detect (like background or liver in medical imaging).

1 (Less represented class)	0.6284 . This suggests your model struggles more here. Could be due to fewer samples, ambiguous boundaries, or class imbalance.

2 (Another minor or difficult class)	0.6875	 Slightly better than Class 1, but still not ideal. There's room for improvement — maybe the object is smaller, has fuzzy edges, or is similar in intensity to others.

In [14]:
def dice_score_foreground(preds, targets, epsilon=1e-6):
    preds = torch.argmax(preds, dim=1)  # (B, H, W)
    targets = torch.argmax(targets, dim=1)  # (B, H, W)

    dice_scores = []

    for cls in [1, 2]:  # Only foreground classes
        pred_cls = (preds == cls).float()
        target_cls = (targets == cls).float()
        
        intersection = (pred_cls * target_cls).sum()
        union = pred_cls.sum() + target_cls.sum()
        
        dice = (2. * intersection + epsilon) / (union + epsilon)
        dice_scores.append(dice)

    return sum(dice_scores) / len(dice_scores)

# Compute combined Dice for foreground (classes 1 and 2)
total_dice_fg = 0
num_batches = 0

with torch.no_grad():
    for images, masks in test_loader:
        outputs = model(images)
        batch_dice_fg = dice_score_foreground(outputs, masks)
        total_dice_fg += batch_dice_fg.item()
        num_batches += 1

avg_dice_fg = total_dice_fg / num_batches
print(f"Average Dice Score for Foreground (Classes 1 & 2): {avg_dice_fg:.4f}")

Average Dice Score for Foreground (Classes 1 & 2): 0.6579


In [15]:
y_pred_list = []  # List to collect all predictions

with torch.no_grad():
    for images, masks in test_loader:
        outputs = model(images)  # outputs shape: (batch_size, num_classes, H, W)
        # Get predicted class labels by taking argmax across the 'class' dimension
        y_pred_labels = torch.argmax(outputs, dim=1)  # shape: (batch_size, H, W)
        # Store the predicted labels
        y_pred_list.append(y_pred_labels.cpu())
        num_batches += 1

# After the loop, you can concatenate all predictions if needed:
y_pred_all = torch.cat(y_pred_list, dim=0)  #

In [25]:
print(y_pred_all.shape)

torch.Size([501, 128, 128])


In [43]:
# ---- Display some slices ----
num_slices = 501  # Number of slices to display
fig, axes = plt.subplots(num_slices, 3, figsize=(10, num_slices * 3))

for i in range(num_slices):
    slice_idx = i   # Adjust step to view different slices

    # Display CT Image
    axes[i, 0].imshow(X_test[slice_idx, :, :], cmap='gray')
    axes[i, 0].set_title(f"Slice {slice_idx} - CT Image")

    # Display Actual Mask
    axes[i, 1].imshow(Y_test_labels[slice_idx], cmap='jet')
    axes[i, 1].set_title("Actual Mask")

    # # Display Predicted Mask
    axes[i, 2].imshow(y_pred_labels[slice_idx], cmap='jet')
    axes[i, 2].set_title("Predicted Mask")

plt.tight_layout()
plt.show()

In [22]:
# Convert actual one-hot masks to class labels
Y_test_labels = np.argmax(Y_test, axis=-1)  # Shape: (total_slices, 128, 128)
print(Y_test_labels.shape)
print(y_pred_labels.shape)

(501, 128, 128)
torch.Size([5, 128, 128])


In [24]:
print(Y_test.shape)

(501, 128, 128, 3)


In [26]:
import os
import matplotlib.pyplot as plt
import numpy as np

# === Configuration ===
num_slices = 501  # Number of slices to display and save
output_dir_pred = "outputs/Unet++/predicted_masks"
output_dir_actual = "outputs/actual_masks"

# Create both output directories
os.makedirs(output_dir_pred, exist_ok=True)
os.makedirs(output_dir_actual, exist_ok=True)

# === Visualization and Saving ===
for i in range(num_slices):
    slice_idx = i  # Adjust this if needed

    # Save the Predicted Mask
    plt.figure(figsize=(4, 4))
    plt.imshow(y_pred_all[slice_idx], cmap='jet')
    plt.axis('off')
    plt.title(f"Predicted Mask - Slice {slice_idx}")
    save_path_pred = os.path.join(output_dir_pred, f"predicted_slice_{slice_idx}.png")
    plt.savefig(save_path_pred, bbox_inches='tight', pad_inches=0)
    print(f"Saved predicted mask: {save_path_pred}")
    plt.close()

    # Save the Actual Mask
    plt.figure(figsize=(4, 4))
    plt.imshow(Y_test_labels[slice_idx], cmap='jet')
    plt.axis('off')
    plt.title(f"Actual Mask - Slice {slice_idx}")
    save_path_actual = os.path.join(output_dir_actual, f"actual_slice_{slice_idx}.png")
    plt.savefig(save_path_actual, bbox_inches='tight', pad_inches=0)
    print(f"Saved actual mask: {save_path_actual}")
    plt.close()

Saved predicted mask: outputs/Unet++/predicted_masks\predicted_slice_0.png
Saved actual mask: outputs/actual_masks\actual_slice_0.png
Saved predicted mask: outputs/Unet++/predicted_masks\predicted_slice_1.png
Saved actual mask: outputs/actual_masks\actual_slice_1.png
Saved predicted mask: outputs/Unet++/predicted_masks\predicted_slice_2.png
Saved actual mask: outputs/actual_masks\actual_slice_2.png
Saved predicted mask: outputs/Unet++/predicted_masks\predicted_slice_3.png
Saved actual mask: outputs/actual_masks\actual_slice_3.png
Saved predicted mask: outputs/Unet++/predicted_masks\predicted_slice_4.png
Saved actual mask: outputs/actual_masks\actual_slice_4.png
Saved predicted mask: outputs/Unet++/predicted_masks\predicted_slice_5.png
Saved actual mask: outputs/actual_masks\actual_slice_5.png
Saved predicted mask: outputs/Unet++/predicted_masks\predicted_slice_6.png
Saved actual mask: outputs/actual_masks\actual_slice_6.png
Saved predicted mask: outputs/Unet++/predicted_masks\predicted